# edge-tts 배치 2 — 426개 빠알리어 음성 생성

GitHub에서 텍스트 다운로드 → edge-tts 생성 → ZIP 자동 다운로드

In [ ]:
!pip install -q edge-tts
print('설치 완료!')

In [ ]:
# 데바나가리 변환 함수
CONSONANTS = {
    'kh': '\u0916', 'k': '\u0915', 'gh': '\u0918', 'g': '\u0917', '\u1e45': '\u0919',
    'ch': '\u091b', 'c': '\u091a', 'jh': '\u091d', 'j': '\u091c', '\u00f1': '\u091e',
    '\u1e6dh': '\u0920', '\u1e6d': '\u091f', '\u1e0dh': '\u0922', '\u1e0d': '\u0921', '\u1e47': '\u0923',
    'th': '\u0925', 't': '\u0924', 'dh': '\u0927', 'd': '\u0926', 'n': '\u0928',
    'ph': '\u092b', 'p': '\u092a', 'bh': '\u092d', 'b': '\u092c', 'm': '\u092e',
    'y': '\u092f', 'r': '\u0930', 'l': '\u0932', '\u1e37': '\u0933',
    'v': '\u0935', 's': '\u0938', 'h': '\u0939',
}
VOWELS_IND = { '\u0101': '\u0906', 'a': '\u0905', '\u012b': '\u0908', 'i': '\u0907', '\u016b': '\u090a', 'u': '\u0909', 'e': '\u090f', 'o': '\u0913' }
VOWELS_DEP = { '\u0101': '\u093e', 'a': '', '\u012b': '\u0940', 'i': '\u093f', '\u016b': '\u0942', 'u': '\u0941', 'e': '\u0947', 'o': '\u094b' }
VIRAMA = '\u094d'

def pali_to_devanagari(roman):
    result = ''; i = 0; s = roman.lower()
    while i < len(s):
        ch = s[i]
        if ch in ' ,.;:!?-\n\r\t': result += ch; i += 1; continue
        if ch == '\u1e43': result += '\u0902'; i += 1; continue
        three = s[i:i+3]; two = s[i:i+2]
        consonant = None; consumed = 0
        if three in CONSONANTS: consonant = CONSONANTS[three]; consumed = 3
        elif two in CONSONANTS: consonant = CONSONANTS[two]; consumed = 2
        elif ch in CONSONANTS: consonant = CONSONANTS[ch]; consumed = 1
        if consonant:
            i += consumed
            if i < len(s) and s[i] in VOWELS_IND: result += consonant + VOWELS_DEP[s[i]]; i += 1
            else: result += consonant + VIRAMA
            continue
        if ch in VOWELS_IND: result += VOWELS_IND[ch]; i += 1; continue
        result += ch; i += 1
    return result

print('변환 함수 준비')

In [ ]:
# 배치 2: GitHub에서 426개 다운로드 → 생성 → ZIP
import edge_tts, os, json, zipfile, urllib.request
from google.colab import files

VOICE = 'hi-IN-MadhurNeural'
RATE = '-30%'

url = 'https://raw.githubusercontent.com/ReachToWisdom/SuttaLog2/main/tts-batch-2.json'
print(f'다운로드: {url}')
texts = json.loads(urllib.request.urlopen(url).read().decode('utf-8'))
print(f'배치 2: {len(texts)}개')

os.makedirs('audio_mp3', exist_ok=True)

async def speak(text, filename):
    comm = edge_tts.Communicate(text, VOICE, rate=RATE)
    await comm.save(filename)

mapping = {}
errors = []

for i, text in enumerate(texts):
    fname = f'b2_{i:04d}.mp3'
    outpath = f'audio_mp3/{fname}'
    mapping[text] = fname
    if os.path.exists(outpath):
        print(f'[{i+1}/{len(texts)}] skip')
        continue
    try:
        clean = text.replace('\n', ' ').strip()
        deva = pali_to_devanagari(clean)
        await speak(deva, outpath)
        print(f'[{i+1}/{len(texts)}] {deva[:30]}')
    except Exception as e:
        errors.append((text, str(e)))
        del mapping[text]
        print(f'[{i+1}/{len(texts)}] ERR: {e}')

print(f'완료! 성공: {len(mapping)}, 오류: {len(errors)}')

with open('audio_mp3/manifest.json', 'w', encoding='utf-8') as f:
    json.dump(mapping, f, ensure_ascii=False, indent=2)

with zipfile.ZipFile('pali-audio-2.zip', 'w', zipfile.ZIP_DEFLATED) as zf:
    for fn in os.listdir('audio_mp3'):
        zf.write(f'audio_mp3/{fn}', fn)

size = os.path.getsize('pali-audio-2.zip') / (1024*1024)
print(f'pali-audio-2.zip ({size:.1f} MB) - {len(mapping)}개')
files.download('pali-audio-2.zip')

